#### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import pickle
import warnings
import os
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')

# Import KMRF class
from kmrf import KMRF
from KMRF_training_config import *

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


## Load Master Data for Feature Access

In [2]:
# Load master data once to get all available trading dates
master_df_path = Path('data/master_df.csv')
master_df = pd.read_csv(master_df_path, index_col=0, header=[0, 1, 2], parse_dates=True)
master_df.index = pd.to_datetime(master_df.index)

# Get all trading dates from the data
all_trading_dates = master_df.index.tolist()

print(f"Master data loaded")
print(f"Total trading dates: {len(all_trading_dates)}")
print(f"Date range: {all_trading_dates[0].date()} to {all_trading_dates[-1].date()}")

Master data loaded
Total trading dates: 9148
Date range: 1990-01-02 to 2025-10-31


## Configuration

In [3]:
# Configuration
ASSET_CLASS = 'us_equity'
MAX_HORIZON = 21
CLASSIFICATION_TYPE = 'original'  # 'original' for 4-regime, 'adapted' for 3-class

# Path to KAMA_MSR models
BASE_MODEL_PATH = Path('saved_models/KAMA_MSR') / ASSET_CLASS

print(f"Asset Class: {ASSET_CLASS}")
print(f"Max Horizon: {MAX_HORIZON} days")
print(f"Classification Type: {CLASSIFICATION_TYPE}")
print(f"Model Path: {BASE_MODEL_PATH}")

Asset Class: us_equity
Max Horizon: 21 days
Classification Type: original
Model Path: saved_models/KAMA_MSR/us_equity


In [9]:
# Read all files in base model path
model_files = list((BASE_MODEL_PATH / '20181231').glob('*.pkl'))

exists_files = list(Path('data/multi_horizon_predictions').glob('*'))
exists_asset_names= [f.stem.split('multi')[0][:-1].replace('_', ' ') for f in exists_files]

# get asset names from model files
ASSET_NAMES = [f.stem.split('_')[0] for f in model_files]
ASSET_NAMES = list(set(ASSET_NAMES) - set(exists_asset_names))

print(len(ASSET_NAMES))
ASSET_NAMES

6


['iShares China Large-Cap ETF',
 'Vanguard FTSE Developed Markets ETF',
 'Vanguard FTSE Emerging Markets ETF',
 'Vanguard FTSE Europe ETF',
 'iShares MSCI Japan ETF',
 'iShares MSCI India ETF']

## Build and Save Results in Parallelized Loop

In [10]:
def build_multi_horizon_predictions(
    asset_name: str,
    asset_class: str,
    prediction_windows: List[Dict],
    max_horizon: int = 21,
    classification_type: str = 'original',
    verbose: bool = True
) -> pd.DataFrame:
    """
    Build comprehensive multi-horizon predictions DataFrame.
    
    Parameters
    ----------
    asset_name : str
        Name of the asset
    asset_class : str
        Asset class (e.g., 'us_equity')
    prediction_windows : List[Dict]
        List of prediction window definitions
    max_horizon : int, default=21
        Maximum prediction horizon
    classification_type : str, default='original'
        'original' for 4-regime or 'adapted' for 3-class
    verbose : bool, default=True
        Print progress
        
    Returns
    -------
    pd.DataFrame
        MultiIndex DataFrame (date, horizon) with regime probabilities
    """
    all_records = []
    
    total_windows = len(prediction_windows)
    
    for window_idx, window in enumerate(prediction_windows):
        model_end_date = window['model_end_date']
        pred_start = window['pred_start']
        pred_end = window['pred_end']
        
        if verbose:
            print(f"\n{'='*80}")
            print(f"Window {window_idx + 1}/{total_windows}: Model {model_end_date}")
            print(f"Predictions: {pred_start.date()} to {pred_end.date()}")
            print(f"{'='*80}")
        
        try:
            # Initialize KMRF with this model's end date
            kmrf = KMRF(
                asset_name=asset_name,
                asset_class=asset_class,
                classification_type=classification_type,
                end_date=model_end_date,
                use_data_type='master',
                feature_window_size=1,
                feature_asset_classes=[],
                cross_asset_specific=[],
                use_boruta_selection=False,
                use_consensus_selection=False
            )
            
            # Run pipeline (loads data, features, labels, trains model)
            kmrf.pipeline_multi_horizon(max_horizon=max_horizon, verbose=False)
            
            # Get all OOS predictions
            all_oos_preds = kmrf.predict_multi_horizon_all_oos(horizon=max_horizon)
            
            # Filter to only dates within this window
            dates_in_window = [d for d in all_oos_preds.keys() 
                               if pred_start <= d <= pred_end]
            
            if verbose:
                print(f"  Generated predictions for {len(dates_in_window)} dates in window")
            
            # Convert to records
            for date in dates_in_window:
                pred_df = all_oos_preds[date]
                for horizon in pred_df.index:
                    row = {
                        'date': date,
                        'horizon': horizon,
                        'model_end_date': model_end_date
                    }
                    row.update(pred_df.loc[horizon].to_dict())
                    all_records.append(row)
            
            if verbose:
                print(f"  ✓ Window complete. Total records so far: {len(all_records)}")
                
        except Exception as e:
            print(f"  ✗ Error processing window {model_end_date}: {str(e)}")
            continue
    
    # Create DataFrame
    if len(all_records) == 0:
        raise ValueError("No predictions generated")
    
    predictions_df = pd.DataFrame(all_records)
    predictions_df = predictions_df.set_index(['date', 'horizon'])
    predictions_df = predictions_df.sort_index()
    
    return predictions_df

def get_prediction_windows(model_dates: List[str], all_dates: List[pd.Timestamp]) -> List[Dict]:
    """
    Determine the prediction window for each model.
    
    Each model trained up to end_date T is used for predictions starting from T at close
    (so horizon 1 predicts T+1) until the day before the next model's end_date.
    
    Parameters
    ----------
    model_dates : List[str]
        Sorted list of model end dates (YYYYMMDD format)
    all_dates : List[pd.Timestamp]
        All available trading dates
        
    Returns
    -------
    List[Dict]
        List of dicts with 'model_end_date', 'pred_start', 'pred_end' keys
    """
    windows = []
    all_dates_set = set(all_dates)
    
    for i, model_date in enumerate(model_dates):
        model_end = pd.to_datetime(model_date)
        
        # Prediction starts on day after model date end (with 1-day lagged features) (so horizon 1 = prediction start)
        # Find the model_end date in trading dates, or closest before it
        pred_start = None
        # for d in reversed(all_dates):
        #     if d <= model_end:
        #         pred_start = d
        #         break
        for d in all_dates:
            if d > model_end:
                pred_start = d
                break
        
        if pred_start is None:
            continue
        
        # Prediction ends on the next model's end date (or last available date)
        if i + 1 < len(model_dates):
            next_model_end = pd.to_datetime(model_dates[i + 1])
            pred_end = next_model_end
            # pred_end = None
            # for d in reversed(all_dates):
            #     if d < next_model_end:
            #         pred_end = d
            #         break
        else:
            # Last model - predict until last available date
            pred_end = all_dates[-1]
        
        if pred_end is not None and pred_start <= pred_end:
            windows.append({
                'model_end_date': model_date,
                'pred_start': pred_start,
                'pred_end': pred_end
            })
    
    return windows


In [11]:
# get all model dates
model_date_folders = list(BASE_MODEL_PATH.glob('*'))
model_dates = sorted([f.name for f in model_date_folders])[1:]

In [12]:
# get_prediction_windows(model_dates,
#                                 all_dates = [d for d in all_trading_dates if d > pd.Timestamp(model_dates[-12*4])])

In [13]:
# TODO: Why is this failing for newly added assets?

from joblib import Parallel, delayed
N_JOBS = 6

def process_asset(asset_name, asset_class, prediction_windows, max_horizon, classification_type):
    """Process a single asset and save predictions."""
    try:
        print(f"\n{'#'*80}")
        print(f"Processing: {asset_name}")
        print(f"{'#'*80}")
        
        multi_horizon_predictions = build_multi_horizon_predictions(
            asset_name=asset_name,
            asset_class=asset_class,
            prediction_windows=prediction_windows,
            max_horizon=max_horizon,
            classification_type=classification_type,
            verbose=False
        )

        output_dir = Path('data/multi_horizon_predictions')
        output_dir.mkdir(parents=True, exist_ok=True)
        asset_name_clean = asset_name.replace(' ', '_')

        pickle_path = output_dir / f"{asset_name_clean}_multi_horizon_predictions_{classification_type}.pkl"
        multi_horizon_predictions.to_pickle(pickle_path)

        print(f"✓ {asset_name}: Saved to {pickle_path}")
        print(f"  File size: {pickle_path.stat().st_size / 1024**2:.2f} MB")
        return asset_name, True, None
    except Exception as e:
        print(f"✗ {asset_name}: Error - {str(e)}")
        return asset_name, False, str(e)

# Get prediction windows once (same for all assets)
all_prediction_windows = get_prediction_windows(model_dates, all_trading_dates)

print(f"Building multi-horizon predictions for {len(ASSET_NAMES)} assets")
print(f"Each asset will process {len(all_prediction_windows)} model windows")
print(f"Running in parallel...\n")

# Run in parallel
results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(process_asset)(
        asset_name=asset_name,
        asset_class=ASSET_CLASS,
        prediction_windows=all_prediction_windows,
        max_horizon=MAX_HORIZON,
        classification_type=CLASSIFICATION_TYPE
    )
    for asset_name in ASSET_NAMES
)

# Summary
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
successful = [r[0] for r in results if r[1]]
failed = [(r[0], r[2]) for r in results if not r[1]]
print(f"✓ Successfully processed: {len(successful)} assets")
if failed:
    print(f"✗ Failed: {len(failed)} assets")
    for name, error in failed:
        print(f"  - {name}: {error}")

Building multi-horizon predictions for 6 assets
Each asset will process 82 model windows
Running in parallel...



[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.



################################################################################
Processing: Vanguard FTSE Europe ETF
################################################################################
KMRF model initialized
  Asset: Vanguard FTSE Europe ETF
  Asset class: us_equity
  Classification type: original
  Training end date: 20181231
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20181231
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False

KMRF MULTI-HORIZON PIPELINE FOR Vanguard FTSE Europe ETF
Asset Class: us_equity
Classification Type: original
Data Type: master
Max Prediction Horizon: 21 days
Featur

[Parallel(n_jobs=6)]: Done   2 out of   6 | elapsed: 46.1min remaining: 92.2min



Step 4: Using Original 4-Regime Labels
  (LV Bullish=0, LV Bearish=1, HV Bullish=2, HV Bearish=3)
  Labels Shape: (5030,)

Step 5: Cleaning Features
  Features date range: 2005-03-11 00:00:00 to 2025-10-31 00:00:00
  Labels date range: 2005-03-11 00:00:00 to 2025-03-07 00:00:00
  Note: Keeping all feature dates (including validation/test periods)
  Features cleaned and aligned

Step 6: Using standard single-step features (feature_window_size=1)

Step 7: Splitting Data
  Calculating validation/test split dates...
  Calculated split dates:
    Training end   : 2025-03-07
    Validation     : 2025-03-10 to 2025-07-07 (82 samples)
    Test           : 2025-07-08 to 2025-10-31 (83 samples)

  Train/Validation/Test Split:
    Training   : Before 2025-03-10
    Validation : 2025-03-10 to 2025-07-07
    Test       : 2025-07-08 to 2025-10-31

  Split sizes:
    Training   : 4778 samples with labels (2006-03-13 to 2025-03-07)
    Validation : 82 samples (features only, no labels) (2025-03-10 to

[Parallel(n_jobs=6)]: Done   3 out of   6 | elapsed: 49.1min remaining: 49.1min



MULTI-HORIZON TRAINING COMPLETE
Trained 21 models (horizons 1 to 21)
Use predict_multi_horizon() for multi-step predictions

MULTI-HORIZON PIPELINE COMPLETE
✓ Trained 21 models (horizons 1 to 21)
✓ Use predict_multi_horizon() for single-date multi-step predictions
✓ Use predict_multi_horizon_all_oos() for all OOS predictions
✓ Use save_model() to persist trained models


Generating multi-horizon predictions for all OOS data...
  Horizon: 21 days
  Total dates: 18
  Date range: 2025-10-08 00:00:00 to 2025-10-31 00:00:00
  Processed 18/18 dates
✓ Generated multi-horizon predictions for 18 OOS dates
✓ Vanguard FTSE Emerging Markets ETF: Saved to data/multi_horizon_predictions/Vanguard_FTSE_Emerging_Markets_ETF_multi_horizon_predictions_original.pkl
  File size: 1.29 MB


[Parallel(n_jobs=6)]: Done   4 out of   6 | elapsed: 49.2min remaining: 24.6min



MULTI-HORIZON TRAINING COMPLETE
Trained 21 models (horizons 1 to 21)
Use predict_multi_horizon() for multi-step predictions

MULTI-HORIZON PIPELINE COMPLETE
✓ Trained 21 models (horizons 1 to 21)
✓ Use predict_multi_horizon() for single-date multi-step predictions
✓ Use predict_multi_horizon_all_oos() for all OOS predictions
✓ Use save_model() to persist trained models


Generating multi-horizon predictions for all OOS data...
  Horizon: 21 days
  Total dates: 501
  Date range: 2023-11-02 00:00:00 to 2025-10-31 00:00:00
  Processed 100/501 dates
  Processed 100/501 dates
  Processed 200/501 dates
  Processed 200/501 dates
  Processed 300/501 dates
  Processed 300/501 dates
  Processed 400/501 dates
  Processed 400/501 dates
  Processed 500/501 dates
  Processed 501/501 dates
✓ Generated multi-horizon predictions for 501 OOS dates
KMRF model initialized
  Asset: iShares MSCI Japan ETF
  Asset class: us_equity
  Classification type: original
  Training end date: 20231201
  Test end date

[Parallel(n_jobs=6)]: Done   6 out of   6 | elapsed: 88.0min finished
